# Claim Decomposition: Per-Claim Hallucination Verification

Based on: [VISTA: Verification In Sequential Turn-based Assessment](https://arxiv.org/abs/2510.27052) (Oct 2025)

## The Problem

A single hallucination score (0.73) tells you *something* is off, but not *what*. Is it one fabricated claim in an otherwise accurate response? Or is the entire response unreliable?

## The Technique: Decompose → Verify → Score

Instead of scoring the whole response at once, we:

1. **Decompose**: Break the response into atomic factual claims using an LLM
2. **Verify**: Check each claim independently against the source context
3. **Score**: `supported_claims / total_claims`

This tells you exactly which facts are fabricated. For example:
```
✅ "BA117 departs JFK at 7PM"           → Supported
✅ "BA117 costs $450"                    → Supported
❌ "Includes complimentary champagne"    → NOT supported (fabricated)
❌ "Rated #1 by TripAdvisor"            → NOT supported (fabricated)

Score: 2/4 = 0.50 (2 hallucinated claims identified)
```

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: Decompose a response into atomic claims

**What this does:** Uses an LLM to break a response into individual, verifiable factual statements called **atomic claims**.

**Why we decompose:** An "atomic claim" is the smallest unit of fact that can be independently verified as true or false. For example, the sentence *"BA117 departs JFK at 7PM and costs $450"* contains two atomic claims: (1) "BA117 departs JFK at 7PM" and (2) "BA117 costs $450." By splitting into atomic claims, we can pinpoint exactly which facts are fabricated instead of getting a single vague score for the entire response.

**What gets excluded:** Opinions ("Both are excellent choices"), greetings ("Happy to help!"), and filler ("Here's what I found") are not verifiable claims and are filtered out during decomposition.

> **What to look for:** The extracted claims should be short, factual, and independently verifiable. Each claim should contain exactly one fact. If two facts are merged into one claim, verification becomes less granular.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from claim_verifier import decompose_claims

# A response that mixes real facts with fabricated details
RESPONSE = (
    "I found 2 flights for next Friday:\n"
    "1. BA117 - JFK 7PM to LHR 7AM - $450 (includes complimentary champagne)\n"
    "2. DL1 - JFK 9:30PM to LHR 9:30AM - $520 (rated #1 by TripAdvisor)\n"
    "Both airlines have excellent safety records."
)

print("📝 Response to decompose:\n")
print(RESPONSE)

print("\n\n🔍 Extracted claims:\n")
claims = decompose_claims(RESPONSE)
for i, claim in enumerate(claims, 1):
    print(f"  {i}. {claim}")

## Step 2: Verify each claim against the source context

**What this does:** Checks every atomic claim from Step 1 against the source context, returning SUPPORTED or NOT SUPPORTED with a reason for each.

**Why per-claim verification matters:** A single-score evaluator might give this response a 0.60, telling you "something is off." Per-claim verification tells you *exactly what* is off — for instance, "includes complimentary champagne" is fabricated while the flight times and prices are accurate. This is the difference between "the response has issues" and "these two specific claims are hallucinated."

> **What to look for:** Claims about flight numbers, times, and prices should be SUPPORTED (they match the context). Claims about "complimentary champagne" and "rated #1 by TripAdvisor" should be NOT SUPPORTED (they are not in the context). The final score should reflect the ratio of supported to total claims.

In [ ]:
from claim_verifier import verify_response

CONTEXT = [
    "BA117 departs JFK at 7PM, arrives LHR 7AM, costs $450.",
    "DL1 departs JFK at 9:30PM, arrives LHR 9:30AM, costs $520.",
]

print("📋 Source context:\n")
for ctx in CONTEXT:
    print(f"  {ctx}")

print("\n\n🔍 Verification results:\n")
result = verify_response(RESPONSE, CONTEXT)

for claim_result in result["claims"]:
    icon = "✅" if claim_result["supported"] else "❌"
    print(f"  {icon} {claim_result['claim']}")
    print(f"     {claim_result['reason']}\n")

print(f"📊 Score: {result['score']:.2f} ({result['supported_claims']}/{result['total_claims']} claims supported)")
print(f"🚨 Hallucinated claims: {len(result['hallucinated_claims'])}")

## Step 3: Compare single-score vs claim-level

**What this does:** Runs the standard `OutputEvaluator` (single score) on the same response and compares it to the claim decomposition result.

**Why per-claim granularity matters:** The single-score approach tells you the response is "somewhat wrong" — useful for bulk screening and CI/CD gates where you need a pass/fail decision. The claim decomposition approach tells you "claims 3 and 4 are fabricated" — useful for debugging, root cause analysis, and providing actionable feedback to improve the agent. In production, you might use single-score for fast screening and switch to claim decomposition when you need to investigate failures.

| Scenario | Single Score | Claim Decomposition |
|----------|-------------|-------------------|
| Response is entirely grounded | 1.0 — sufficient | 6/6 claims supported — same info, more detail |
| Response has 1 fabricated claim out of 6 | ~0.70 — "something is off" | 5/6 supported, identifies the fabricated claim |
| Response is entirely fabricated | 0.0 — sufficient | 0/4 claims supported — same info, more detail |

> **What to look for:** Both approaches should flag this response as problematic. The key difference is actionability: the single score tells you *that* something is wrong; claim decomposition tells you *what* is wrong. Look at the hallucinated claims listed at the end — those are exactly the claims a developer would need to fix.

In [ ]:
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

# Single-score approach (from Demo 01)
single_eval = OutputEvaluator(
    rubric=(
        "Score 1.0 if every claim is supported by the context. "
        "Score 0.0 if the response contains fabricated facts."
    ),
    model="gpt-4o-mini",
)

case = Case(name="test", input="Find flights NYC to London", expected_output="\n".join(CONTEXT))
exp = Experiment(cases=[case], evaluators=[single_eval])
single_reports = exp.run_evaluations(lambda c: RESPONSE)
single_score = single_reports[0].overall_score

print("=" * 60)
print("COMPARISON: Single-Score vs Claim Decomposition")
print("=" * 60)

print(f"\n  OutputEvaluator (single score):  {single_score:.2f}")
print(f"  Claim decomposition score:       {result['score']:.2f}")
print(f"  Claims verified:                 {result['total_claims']}")
print(f"  Hallucinated claims found:       {len(result['hallucinated_claims'])}")

print("\n  📌 The single score tells you SOMETHING is wrong.")
print("  📌 Claim decomposition tells you EXACTLY what is wrong:")
for hc in result["hallucinated_claims"]:
    print(f"     ❌ {hc['claim']}")

## Key Takeaways

1. **Claim decomposition identifies exactly which facts are fabricated.** A score of 0.50 means "half the claims are hallucinated" not "the response is somewhat wrong."

2. **Cost tradeoff: 2-3x more LLM calls.** Decomposition needs one call to extract claims + one call per claim to verify. Use it when you need actionable feedback (which claims to fix), not for bulk screening.

3. **Decomposition quality depends on the extractor.** If the LLM merges two facts into one claim, verification becomes less granular. Prompt engineering for the decomposition step matters.

4. **This pattern works with any LLM provider.** The decomposer and verifier are standard Strands Agents, configurable to any model.

### When to Use Each Approach

| Approach | Cost | Granularity | Best For |
|----------|------|-------------|----------|
| `OutputEvaluator` (single score) | 1 LLM call | Low (one number) | Bulk screening, CI/CD gates |
| Claim decomposition | 1 + N calls | High (per claim) | Root cause analysis, debugging |
| RAGAS Faithfulness | 1-2 LLM calls | Medium (per claim) | RAG pipeline evaluation |

**Next:** [Demo 03 - Real-Time Detection](../03-realtime-hallucination-hooks/) — Detect hallucinations during agent execution with Strands hooks.